given polish_dataset_7_no_chain/merged_is_only


In [1]:
chunk_ids = [
    "classified_information_623",
    "education_law_1055",
    "education_law_1522",
    "financing_education_652",
    "financing_education_950",
    "healthcare_3946",
    "highereducation_science_1331",
    "highereducation_science_1507",
    "highereducation_science_793",
    "military_1582",
    "military_2037",
    "military_488",
    "police_1208",
    "police_1219",

]
# chunk that were clearly tt

In [2]:
import os
import json
from tqdm import tqdm
import pandas as pd
from FlagEmbedding import BGEM3FlagModel
import numpy as np

# Path to queries.jsonl
queries_path = '../data/processed/polish_dataset_7_no_chain/merged_r6/queries.jsonl'

# Provided chunk_ids
chunk_ids_set = set(chunk_ids)

# Load queries
queries = []
with open(queries_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)
        if entry.get('chunk_id') in chunk_ids_set:
            queries.append(entry)

print(f"Loaded {len(queries)} queries with matching chunk_ids.")

# Inspect a sample
for q in queries[:2]:
    print(q['chunk_id'], q.get('chunk', '')[:60], q.get('context_chunks', '')[:60])


/Users/m-proj/repositories/long-context-retrieval/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 14 queries with matching chunk_ids.
classified_information_623 1) upłynął okres jego ważności, o którym mowa w ust. 2; 2. Świadectwo potwierdzające zdolność do ochrony informacji 
police_1208 10. Przełożony, o którym mowa w art. 32 ust. 1, lub osoba pr 1 67) W brzmieniu ustalonym przez art. 2 pkt 7 ustawy, o któ


In [3]:
# Load BGE-M3 model (sparse and dense)
device = 'mps'  # or 'cuda' if available
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device=device)

# Helper: cosine similarity for dense vectors
def cosine_dense(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2) + 1e-9)

# Helper: cosine similarity for sparse vectors
def sparse_norm(sv: dict) -> float:
    return np.sqrt(sum(w**2 for w in sv.values()))
def cosine_sparse(sv1: dict, sv2: dict) -> float:
    dot = sum(sv1[k] * sv2[k] for k in sv1 if k in sv2)
    return dot / (sparse_norm(sv1) * sparse_norm(sv2) + 1e-9)


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 285326.80it/s]


In [4]:
# Compute similarities for each query
results = []
for q in tqdm(queries):
    chunk_id = q['chunk_id']
    query = q.get('query', '')
    chunk = q.get('chunk', '')
    context = q.get('context_chunks', '')
    if not query or not chunk or not context:
        continue
    # Get both sparse and dense embeddings in one call
    emb_query = model.encode([query], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    emb_chunk = model.encode([chunk], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    emb_context = model.encode([context], return_dense=True, return_sparse=True, return_colbert_vecs=False)
    # Scores between query and chunk
    lexical_overlap_chunk = model.compute_lexical_matching_score(
        emb_query['lexical_weights'][0], emb_chunk['lexical_weights'][0])
    sparse_cosine_chunk = cosine_sparse(
        emb_query['lexical_weights'][0], emb_chunk['lexical_weights'][0])
    dense_cosine_chunk = cosine_dense(
        emb_query['dense_vecs'][0], emb_chunk['dense_vecs'][0])
    # Scores between query and context_chunks
    lexical_overlap_context = model.compute_lexical_matching_score(
        emb_query['lexical_weights'][0], emb_context['lexical_weights'][0])
    sparse_cosine_context = cosine_sparse(
        emb_query['lexical_weights'][0], emb_context['lexical_weights'][0])
    dense_cosine_context = cosine_dense(
        emb_query['dense_vecs'][0], emb_context['dense_vecs'][0])
    results.append({
        'chunk_id': chunk_id,
        'lexical_overlap_chunk': lexical_overlap_chunk,
        'sparse_cosine_chunk': sparse_cosine_chunk,
        'dense_cosine_chunk': dense_cosine_chunk,
        'lexical_overlap_context': lexical_overlap_context,
        'sparse_cosine_context': sparse_cosine_context,
        'dense_cosine_context': dense_cosine_context,
        'query': query,
        'chunk': chunk,
        'context_chunk': context
    })

results_df = pd.DataFrame(results)
results_df.head()


100%|██████████| 14/14 [00:08<00:00,  1.60it/s]
/Users/m-proj/repositories/long-context-retrieval/.venv/lib/python3.11/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,chunk_id,lexical_overlap_chunk,sparse_cosine_chunk,dense_cosine_chunk,lexical_overlap_context,sparse_cosine_context,dense_cosine_context,query,chunk,context_chunk
0,classified_information_623,0.022156,0.088562,0.495117,0.212891,0.519043,0.768066,Po upływie jakiego okresu ważności wygasa świa...,"1) upłynął okres jego ważności, o którym mowa ...",2. Świadectwo potwierdzające zdolność do ochro...
1,police_1208,0.261230,0.638672,0.764648,0.038391,0.060120,0.481201,Kto może cofnąć polecenie wykonywania służby w...,"10. Przełożony, o którym mowa w art. 32 ust. 1...",1 67) W brzmieniu ustalonym przez art. 2 pkt 7...
2,financing_education_950,0.008667,0.018967,0.621094,0.142700,0.149902,0.669922,Which textbooks provided to primary schools fo...,"3. Podręczniki, o których mowa w ust. 1, są do...",Art. 119. 1. Na lata szkolne 2017/2018 i 2018/...
3,healthcare_3946,0.255615,0.442871,0.704102,0.217041,0.583984,0.636719,"Jakie podmioty, którym Fundusz powierzył wykon...","3. W razie stwierdzenia, na podstawie uzyskany...","3) podmiotów, którym Fundusz powierzył wykonyw..."
4,highereducation_science_793,0.272461,0.505371,0.681152,0.242432,0.437012,0.650391,Jakiego zaświadczenia z ośrodka pomocy społecz...,5 28) W brzmieniu ustalonym przez art. 1 pkt 1...,4 27) W tym brzmieniu obowiązuje do wejścia w ...


In [15]:
results_df.sort_values('chunk_id')[['chunk_id', 'sparse_cosine_chunk', 'sparse_cosine_context', 'dense_cosine_chunk', 'dense_cosine_context']]

/Users/m-proj/repositories/long-context-retrieval/.venv/lib/python3.11/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,chunk_id,sparse_cosine_chunk,sparse_cosine_context,dense_cosine_chunk,dense_cosine_context
0,classified_information_623,0.088562,0.519043,0.495117,0.768066
6,education_law_1055,0.162109,0.466553,0.617676,0.628418
7,education_law_1522,0.580078,0.143311,0.620117,0.641602
12,financing_education_652,0.605469,0.312012,0.759277,0.602539
2,financing_education_950,0.018967,0.149902,0.621094,0.669922
3,healthcare_3946,0.442871,0.583984,0.704102,0.636719
11,highereducation_science_1331,0.485596,0.000000,0.694824,0.406494
8,highereducation_science_1507,0.551758,0.438721,0.764160,0.791992
4,highereducation_science_793,0.505371,0.437012,0.681152,0.650391
9,military_1582,0.125854,0.151001,0.627441,0.585938


In [6]:
# Show summary statistics and a sample
print(results_df.describe())

       lexical_overlap_chunk  sparse_cosine_chunk  dense_cosine_chunk  \
count              14.000000            14.000000           14.000000   
mean                0.162476             0.382812            0.668457   
std                 0.103027             0.215698            0.076294   
min                 0.008667             0.018967            0.495117   
25%                 0.061249             0.183685            0.620361   
50%                 0.185913             0.464233            0.682129   
75%                 0.250885             0.572266            0.719116   
max                 0.308838             0.638672            0.764648   

       lexical_overlap_context  sparse_cosine_context  dense_cosine_context  
count                14.000000              14.000000             14.000000  
mean                  0.167084               0.330322              0.643555  
std                   0.101967               0.195923              0.107544  
min                   0.000000